# VectorDrugMatch - comparison against published-baseline methods (GPU)

Runs the four standard knowledge-graph-embedding baselines (**TransE, DistMult, ComplEx, RotatE**) - the model class the drug-repurposing
literature, including the TxGNN paper, benchmarks against - on **exactly the same** splits, leakage-safe graph and candidate pairs as
VectorDrugMatch, and also measures VectorDrugMatch under the **zero-shot-disease protocol** that published methods use.

This is a **post-hoc comparison**. The deployed model was selected and frozen before these runs
(`reports/final_audit_20261117/FROZEN_SELECTION.json`) and nothing here changes it. The locked holdout is scored for comparison only;
no model is selected on it.

Code pinned to `33dfb04222ca8af3069f3aa857c3094796028394`. Outputs go to a NEW Drive folder `MyDrive/VDM_Outputs/comparison_20261118/`. Total about 1.5-2 h on a T4.
Run cells top to bottom; every stage is resumable (`--resume` skips finished runs).

## Step 1 - GPU, code, libraries

In [ ]:
import torch; assert torch.cuda.is_available(), 'Runtime > Change runtime type > T4 GPU'
print(torch.cuda.get_device_name(0))
from google.colab import drive; drive.mount('/content/drive')
%cd /content
!rm -rf VectorDrugMatch && git clone -q -b exp/final-audit-20261117 https://github.com/MUKUL-PRASAD-SIGH/VectorDrugMatch
%cd /content/VectorDrugMatch
!git checkout -q 33dfb04222ca8af3069f3aa857c3094796028394 && git log -1 --oneline
%pip install -q torch-geometric

## Step 2 - Data and the committed split files, verified by SHA-256 (stop on mismatch)

In [ ]:
import hashlib, json, shutil, os
D = '/content/drive/MyDrive/VDM_Data/frozen_processed/primekg'
assert os.path.isdir(D), 'MyDrive/VDM_Data/frozen_processed/primekg is missing'
shutil.copytree(D, 'data/processed/primekg', dirs_exist_ok=True)
man = json.load(open('Docs/release/vdm_v1_candidate_manifest.json'))['files']
want = {f['path']: f['sha256'] for f in man if f['path'].startswith('data/processed/primekg/') and f['path'].endswith(('.csv','.npy','.npz'))}
bad = [p for p, h in want.items() if hashlib.sha256(open(p,'rb').read()).hexdigest() != h]
assert not bad, f'DATA MISMATCH {bad}'
P = 'data/processed/primekg/splits_final_20261117'
shutil.rmtree(P, ignore_errors=True); shutil.copytree('reports/final_audit_20261117/protocol_files', P)
ref = json.load(open('reports/final_audit_20261117/protocol_manifest_committed.json'))
ok = all(hashlib.sha256(open(f'{P}/{k}','rb').read()).hexdigest() == v['sha256'] for k, v in ref['files'].items())
ok &= hashlib.sha256(open(f'{P}/holdout_LOCKED.npz','rb').read()).hexdigest() == ref['holdout']['sha256']
Q = 'data/processed/primekg/splits_disease_20261118'
shutil.rmtree(Q, ignore_errors=True); shutil.copytree('reports/final_audit_20261117/disease_protocol_files', Q)
dref = json.load(open(f'{Q}/manifest.json'))
ok &= hashlib.sha256(open(f'{Q}/disease_split.npz','rb').read()).hexdigest() == dref['file_sha256']
assert ok, 'SPLIT FILE MISMATCH - STOP'
print('DATA AND ALL SPLIT FILES VERIFIED')
N = 'comparison_20261118'; O = f'/content/drive/MyDrive/VDM_Outputs/{N}'
os.makedirs(O, exist_ok=True); os.makedirs('outputs', exist_ok=True)
if not os.path.islink('outputs/final_audit_20261117'): os.symlink(O, 'outputs/final_audit_20261117')
print('outputs ->', os.path.realpath('outputs/final_audit_20261117'))

## Stage 1 (run first) - KGE baselines trained on `final_train`, scored on the locked holdout (4 runs, about 25 min)
**The key comparison - run this one first.** Same training data as the deployed model `vdm-1.0.0`, so these holdout numbers are directly comparable with its headline result.

In [ ]:
!python -u scripts/final_audit/train_kge.py --models TransE DistMult ComplEx RotatE --splits final_train --seeds 42 --steps 12000 --eval-every 1000 --patience 4 --eval-holdout --resume --out outputs/final_audit_20261117/kge 2>&1 | grep -E 'DONE|early stop|FAIL'

## Stage 2 - the zero-shot-DISEASE protocol (the published setting)
Diseases are the held-out unit here (TxGNN's `complex_disease` setting), so these numbers are **not** comparable with the drug-disjoint
results and must never be mixed with them. About 20 min.

In [ ]:
!python -u scripts/final_audit/train_gcn_protocol.py --name L3_combo --splits disease_split --seeds 21 42 52 --layers 4 --emb 128 --head 64 --dropout 0.4 --wd 0.0005 --epochs 200 --patience 20 --resume --out outputs/final_audit_20261117/disease_protocol 2>&1 | grep -E 'DONE|early stop|FAIL'

In [ ]:
!python -u scripts/final_audit/dev_cv_baselines.py --splits disease_split --skip-rf --out outputs/final_audit_20261117/disease_protocol 2>&1 | grep -E '^disease_split|Error'

In [ ]:
!python -u scripts/final_audit/train_kge.py --models TransE DistMult ComplEx RotatE --splits disease_split --seeds 42 --steps 12000 --eval-every 1000 --patience 4 --resume --out outputs/final_audit_20261117/kge 2>&1 | grep -E 'DONE|early stop|FAIL'

## Stage 3 (run last) - KGE baselines on the 5 development folds (20 runs, about 60 min)
Fold-to-fold variability, on the same folds and candidate pairs as every VectorDrugMatch development number.

In [ ]:
!python -u scripts/final_audit/train_kge.py --models TransE DistMult ComplEx RotatE --splits cv_fold0 cv_fold1 cv_fold2 cv_fold3 cv_fold4 --seeds 42 --steps 12000 --eval-every 1000 --patience 4 --resume --out outputs/final_audit_20261117/kge 2>&1 | grep -E 'DONE|early stop|FAIL'

## Step 4 - check the device and the audits, then save. Wait for `SAVED`.

In [ ]:
import json, glob, collections, os
from google.colab import drive
rc = [json.load(open(f)) for f in glob.glob('outputs/final_audit_20261117/**/run_config.json', recursive=True)]
print(len(rc), 'runs |', dict(collections.Counter(str(r.get('gpu')) for r in rc)))
assert all(r.get('gpu') for r in rc), 'some runs did not use the GPU'
au = [json.load(open(f))['all_pass'] for f in glob.glob('outputs/final_audit_20261117/**/leakage_audit.json', recursive=True)]
print('leakage audits:', len(au), 'all pass:', all(au)); assert all(au)
Z = f'/content/drive/MyDrive/VDM_Outputs/{N}_results.zip'
!cd outputs && rm -f {Z} && zip -rq {Z} final_audit_20261117 -x '*/model.pt' -x '*/model.pth'
print('zip MB:', round(os.path.getsize(Z)/1e6, 1))
drive.flush_and_unmount(); print('SAVED - download', N + '_results.zip from MyDrive/VDM_Outputs')